# Etapa 4 — Explicabilidade (SHAP) e comparação entre eras

Esta etapa responde à pergunta central do trabalho: **quais variáveis o modelo usa para prever
o vencedor, e o peso delas mudou entre a Era Clássica e a Era Moderna?**

Os **valores SHAP** (valores de Shapley) decompõem a previsão de cada partida na contribuição de
cada variável: quanto ela empurrou a probabilidade de vitória do mandante para cima ou para
baixo em relação à previsão média do modelo (o valor-base). Somando o valor-base às
contribuições de todas as variáveis, obtém-se exatamente a probabilidade prevista para aquela
partida.

| | Arquivo | Conteúdo |
|---|---|---|
| Entrada | `modelos/<era>_<algoritmo>.joblib` | modelo final de cada era (etapa 3) |
| Entrada | `dados/processados/features_jogos.parquet`, `resultados/previsoes.parquet` | etapas 2 e 3 |
| Saída | `resultados/shap_valores.npz` | valores SHAP de todas as partidas |
| Saída | `resultados/shap_por_feature.csv`, `shap_por_familia.csv`, `shap_sementes.csv` | importâncias e comparações |
| Saída | `resultados/auc_univariada_temporada.csv` | trajetória temporada a temporada |
| Saída | `figuras/*.png` | gráficos usados no texto |

Três cuidados tornam a comparação entre eras legítima:

1. **Mesmo algoritmo e mesmos hiperparâmetros nas duas eras**, garantidos pela etapa 3 e
   conferidos aqui.
2. **Fatia, e não magnitude.** Compara-se a fração que cada variável ocupa da importância total
   da era, porque a magnitude dos valores SHAP depende da escala das previsões de cada modelo.
3. **Agregação por família.** As janelas de 5 jogos, 10 jogos e temporada de uma mesma métrica
   são fortemente correlacionadas, e o SHAP divide o crédito entre colunas correlacionadas;
   somar a família mede a importância real da métrica.

A execução completa leva cerca de 30 minutos, a maior parte no teste de sementes (seção 4).

In [ ]:
from pathlib import Path
import gc, json, re, time

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from joblib import load
from scipy.stats import rankdata, spearmanr
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 220)

DIR_DADOS = Path("dados/processados")
DIR_RESULTADOS = Path("resultados")
DIR_MODELOS = Path("modelos")
DIR_FIGURAS = Path("figuras")
DIR_FIGURAS.mkdir(exist_ok=True)

SEMENTE = 42
SEMENTES_EXTRAS = [7, 2024]     # teste de estabilidade (seção 4)
N_BOOTSTRAP = 500               # reamostragens de temporadas (seção 4)
B_TRAJETORIA = 1000             # reamostragens de partidas (seção 5)
BLOCO_SHAP = 2000               # partidas por bloco no cálculo do SHAP (memória)
TOP_GRAFICO = 15
FAMILIAS_TRES = ["fg3ar", "fg3_pct", "pct_meia_distancia",
                 "pct_3p_assistidos", "opp_fg3ar", "efg"]
AZUL, LARANJA, CINZA = "#2a78d6", "#eb6834", "#c9c8c2"

dados = pd.read_parquet(DIR_DADOS / "features_jogos.parquet")
meta = json.loads((DIR_RESULTADOS / "metadados_modelagem.json").read_text(encoding="utf-8"))
ALGORITMO = meta["algoritmo_para_shap"]
ERAS = sorted(dados["era"].unique())
CORES_ERA = {ERAS[0]: AZUL, ERAS[1]: LARANJA}

modelos = {era: load(DIR_MODELOS / f"{era}_{ALGORITMO}.joblib") for era in ERAS}
assert len({m["algoritmo"] for m in modelos.values()}) == 1
assert len({repr(m["params"]) for m in modelos.values()}) == 1

print(f"algoritmo: {ALGORITMO} | eras: {ERAS}")
for era, m in modelos.items():
    print(f"  {era}: {len(m['features'])} variáveis | {m['params']}")

## 1. Cálculo dos valores SHAP

Para modelos de árvore, o `TreeExplainer` calcula os valores de Shapley de forma **exata**. A
matriz explicada é a que sai dos passos de preparação do `Pipeline` (`pipeline[:-1]`), ou seja,
exatamente o que o classificador recebe. O cálculo é feito em blocos de 2.000 partidas para
caber na memória.

**Verificação de aditividade.** Em cada partida, valor-base + soma dos valores SHAP tem de
reproduzir a probabilidade prevista pelo modelo. É o que garante que a decomposição explica
este modelo, e não uma aproximação dele. O cálculo é interrompido se a diferença passar de
10⁻⁶.

**Limitação.** Os valores são calculados nas partidas em que o modelo final foi ajustado (a era
inteira). Eles descrevem o que o modelo aprendeu, e as duas eras recebem o mesmo tratamento,
mas não descrevem o comportamento do modelo em partidas nunca vistas.

In [ ]:
def explicar(era):
    pacote = modelos[era]
    pipe, feats = pacote["pipeline"], pacote["features"]
    sub = dados[dados["era"] == era]
    Xt = pipe[:-1].transform(sub[feats])
    modelo = pipe[-1]
    explicador = shap.TreeExplainer(modelo)
    valores, bases = [], []
    for ini in range(0, len(Xt), BLOCO_SHAP):
        saida = explicador(Xt[ini:ini + BLOCO_SHAP], check_additivity=False)
        valores.append(saida.values[..., 1])            # contribuição para a classe 1
        bases.append(saida.base_values[..., 1])
    v, b = np.concatenate(valores), np.concatenate(bases)
    erro = float(np.abs(v.sum(axis=1) + b - modelo.predict_proba(Xt)[:, 1]).max())
    assert erro < 1e-6, f"{era}: SHAP não reproduz a saída do modelo (erro {erro:.2e})"
    return v, b, Xt, sub["game_id"].to_numpy(), sub["temporada"].to_numpy(), feats, erro


shap_vals, base_vals, matrizes, ids_jogos, temporadas, erros_adit = {}, {}, {}, {}, {}, {}
for era in ERAS:
    t0 = time.time()
    v, b, Xt, ids, temps, FEATURES, erro = explicar(era)
    shap_vals[era], base_vals[era], erros_adit[era] = v, b, erro
    matrizes[era], ids_jogos[era], temporadas[era] = Xt, ids, temps
    print(f"{era}: {v.shape[0]:,} partidas × {v.shape[1]} variáveis em {time.time() - t0:.0f}s"
          f" | erro máximo de aditividade {erro:.1e}")

np.savez_compressed(DIR_RESULTADOS / "shap_valores.npz",
                    **{f"valores_{e}": shap_vals[e] for e in ERAS},
                    **{f"base_{e}": base_vals[e] for e in ERAS},
                    features=np.array(FEATURES))
print(f"\nvalores salvos em {DIR_RESULTADOS / 'shap_valores.npz'}")

## 2. Importância de cada variável

A importância de uma variável é a **média do valor absoluto** do seu SHAP: quanto, em média,
ela desloca a previsão, em qualquer direção. A **fatia** é essa média dividida pela soma das
médias de todas as variáveis; soma 100% em cada era e responde à pergunta "de cada 100 unidades
de importância, quantas vão para esta variável".

No gráfico *beeswarm*, cada ponto é uma partida: a posição horizontal é a contribuição da
variável para aquela previsão (à direita, a favor do mandante) e a cor é o valor da variável
(vermelho = alto, azul = baixo).

In [ ]:
importancia = pd.DataFrame({era: np.abs(shap_vals[era]).mean(axis=0) for era in ERAS},
                           index=FEATURES)
fatia = importancia / importancia.sum()

for era in ERAS:
    top = fatia[era].sort_values(ascending=False).head(12)
    print(f"\n=== {era}: 12 variáveis mais importantes ===")
    print(pd.DataFrame({"fatia_%": (top * 100).round(2),
                        "|SHAP| médio": importancia.loc[top.index, era].round(5)}).to_string())

In [ ]:
for era in ERAS:
    exp = shap.Explanation(values=shap_vals[era], base_values=base_vals[era],
                           data=matrizes[era], feature_names=FEATURES)
    shap.plots.beeswarm(exp, max_display=15, show=False)
    plt.title(f"SHAP — Era {era}", fontsize=12)
    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / f"beeswarm_{era}.png", dpi=160, bbox_inches="tight")
    plt.show()

## 3. Comparação entre as eras, por família

As fatias das colunas de cada família são somadas (por exemplo, `dif_j5_ortg`, `dif_j10_ortg`
e `dif_tmp_ortg` formam `ortg`). A tabela traz a fatia de cada família nas duas eras, a
diferença (Moderna − Clássica) e a posição no ranking de cada era.

A **correlação de postos de Spearman** (ρ) mede o quanto as duas eras ordenam as famílias da
mesma forma: vale 1 quando a ordem é idêntica e 0 quando não há relação entre as ordens. O
valor-p associado é a probabilidade de observar uma concordância daquele tamanho se as duas
ordens fossem independentes.

In [ ]:
def familia(coluna):
    # dif_j10_pct_meia_distancia -> pct_meia_distancia
    return re.sub(r"^(j\d+|tmp)_", "", re.sub(r"^(dif|casa|fora)_", "", coluna))


fam = pd.Series([familia(c) for c in FEATURES], index=FEATURES, name="familia")
fatia_fam = fatia.groupby(fam).sum()

comp = pd.DataFrame({"fatia_classica": fatia_fam[ERAS[0]], "fatia_moderna": fatia_fam[ERAS[1]]})
comp["diferenca"] = comp["fatia_moderna"] - comp["fatia_classica"]
comp["rank_classica"] = comp["fatia_classica"].rank(ascending=False).astype(int)
comp["rank_moderna"] = comp["fatia_moderna"].rank(ascending=False).astype(int)

rho_eras, p_eras = spearmanr(comp["fatia_classica"], comp["fatia_moderna"])
print(f"Spearman entre as eras: rho = {rho_eras:.3f} (p = {p_eras:.2g})\n")
tabela = comp.sort_values("diferenca", ascending=False)
print((tabela[["fatia_classica", "fatia_moderna", "diferenca"]] * 100).round(2)
      .join(tabela[["rank_classica", "rank_moderna"]])
      .rename(columns=lambda c: c.replace("fatia_", "fatia_%_").replace("diferenca", "diferenca_pp"))
      .to_string())

In [ ]:
# Gráfico de inclinação: fatia de cada família nas duas eras, lado a lado.
ordem = comp.reindex(comp[["fatia_classica", "fatia_moderna"]].max(axis=1)
                     .sort_values(ascending=False).index).head(TOP_GRAFICO)

fig, ax = plt.subplots(figsize=(8, 7), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
y = np.arange(len(ordem))[::-1]
for yi, (nome, linha) in zip(y, ordem.iterrows()):
    sobe = linha["fatia_moderna"] >= linha["fatia_classica"]
    ax.plot([linha["fatia_classica"] * 100, linha["fatia_moderna"] * 100], [yi, yi],
            color=LARANJA if sobe else CINZA, lw=2, zorder=1, solid_capstyle="round")
    ax.scatter(linha["fatia_classica"] * 100, yi, s=55, color=AZUL, zorder=2)
    ax.scatter(linha["fatia_moderna"] * 100, yi, s=55, color=LARANJA, zorder=2)

ax.set_yticks(y, ordem.index, fontsize=9)
ax.set_xlabel("fatia da importância total da era (%)", fontsize=10, color="#3d3d3a")
ax.set_title("Importância SHAP por família de variável", fontsize=12, color="#1a1a19")
ax.scatter([], [], s=55, color=AZUL, label=ERAS[0])
ax.scatter([], [], s=55, color=LARANJA, label=ERAS[1])
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(axis="x", color="#e5e4df", lw=0.8)
ax.set_axisbelow(True)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(CINZA)
ax.tick_params(colors="#6b6a63", labelsize=9)
plt.tight_layout()
plt.savefig(DIR_FIGURAS / "comparacao_eras.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Quanto confiar nas diferenças entre as eras

Uma diferença de fatia pode vir de duas fontes de variação, e cada uma pede um teste.

### 4.1 Composição das temporadas: *bootstrap* de temporadas

O ***bootstrap*** estima a incerteza de um número repetindo o cálculo em reamostragens dos
dados: sorteia-se, com reposição, um conjunto do mesmo tamanho do original, recalcula-se o
número, e repete-se muitas vezes. A faixa que contém 95% dos valores obtidos é o **intervalo de
confiança de 95%**. Aqui, as 10 temporadas de cada era são sorteadas com reposição 500 vezes e
a diferença de fatia de cada família é recalculada a cada sorteio.

A unidade sorteada é a temporada, e não a partida, porque partidas de uma mesma temporada não
são independentes (compartilham elencos, regras e contexto). Mesmo assim, este teste mede
apenas a variação devida a **quais temporadas** compõem cada era: os modelos não são
reajustados a cada sorteio, então a variação do próprio ajuste do modelo fica fixa. Por isso a
coluna `significativo` não deve ser lida como teste de significância das diferenças, e a seção
4.2 mede a outra fonte.

In [ ]:
rng = np.random.default_rng(SEMENTE)
abs_vals = {era: np.abs(shap_vals[era]) for era in ERAS}
familias = comp.index.to_numpy()
M = np.stack([(fam.to_numpy() == f).astype(float) for f in familias], axis=1)  # soma por família
blocos = {era: [np.flatnonzero(temporadas[era] == t) for t in np.unique(temporadas[era])]
          for era in ERAS}

amostras = np.empty((N_BOOTSTRAP, len(familias)))
for i in range(N_BOOTSTRAP):
    fatias = []
    for era in ERAS:
        sorteadas = rng.integers(0, len(blocos[era]), len(blocos[era]))
        idx = np.concatenate([blocos[era][j] for j in sorteadas])
        m = abs_vals[era][idx].mean(axis=0)
        fatias.append((m / m.sum()) @ M)
    amostras[i] = fatias[1] - fatias[0]            # Moderna - Clássica

ic = pd.DataFrame({"diferenca": comp["diferenca"],
                   "ic_inf": np.percentile(amostras, 2.5, axis=0),
                   "ic_sup": np.percentile(amostras, 97.5, axis=0)}, index=familias)
ic["significativo"] = np.where(ic["ic_inf"] * ic["ic_sup"] > 0, "sim", "não")

print(f"{N_BOOTSTRAP} reamostragens de temporadas: {(ic['significativo'] == 'sim').sum()} de "
      f"{len(ic)} famílias com intervalo que não cruza zero\n")
print((ic.sort_values("diferenca", ascending=False)[["diferenca", "ic_inf", "ic_sup"]] * 100)
      .round(2).join(ic["significativo"]).to_string())

### 4.2 Ajuste do modelo: estabilidade entre sementes

A **semente** é o número que inicializa o gerador aleatório do algoritmo; na floresta aleatória
ela decide quais partidas e variáveis cada árvore sorteia. Trocar a semente produz uma floresta
um pouco diferente com os mesmos dados e hiperparâmetros, e a importância pode se redistribuir
entre variáveis correlacionadas.

O modelo final de cada era é reajustado com mais duas sementes (7 e 2024; o modelo principal usa
42), e a diferença de fatia entre as eras é medida nos três ajustes. Uma diferença só é tratada
como deslocamento entre eras se for estável entre as sementes: se a **amplitude** (maior menos
menor diferença entre as três sementes) for da ordem da própria diferença, ou se o sinal trocar,
a diferença é ruído de ajuste.

In [ ]:
fatias_semente = {SEMENTE: fatia_fam}       # o modelo principal já usa a semente 42
for semente in SEMENTES_EXTRAS:
    colunas = {}
    for era in ERAS:
        t0 = time.time()
        sub = dados[dados["era"] == era]
        pipe = clone(modelos[era]["pipeline"]).set_params(mod__random_state=semente,
                                                          mod__n_jobs=-1)
        pipe.fit(sub[FEATURES], sub["alvo"].to_numpy())
        Xt = pipe[:-1].transform(sub[FEATURES])
        explicador = shap.TreeExplainer(pipe[-1])
        soma = np.zeros(len(FEATURES))
        for ini in range(0, len(Xt), BLOCO_SHAP):
            v = explicador.shap_values(Xt[ini:ini + BLOCO_SHAP], check_additivity=False)
            soma += np.abs(v[..., 1]).sum(axis=0)
        imp = pd.Series(soma / len(Xt), index=FEATURES)
        colunas[era] = (imp / imp.sum()).groupby(fam).sum()
        print(f"semente {semente} | {era}: {time.time() - t0:.0f}s")
        del pipe, explicador, Xt
        gc.collect()
    fatias_semente[semente] = pd.DataFrame(colunas)

dif_sementes = pd.DataFrame({s: f[ERAS[1]] - f[ERAS[0]]
                             for s, f in fatias_semente.items()}) * 100     # pontos percentuais
estabilidade = pd.DataFrame({
    "media_pp": dif_sementes.mean(axis=1),
    "amplitude_pp": dif_sementes.max(axis=1) - dif_sementes.min(axis=1),
    "troca_de_sinal": np.where((dif_sementes > 0).all(axis=1) | (dif_sementes < 0).all(axis=1),
                               "não", "sim"),
}).join(dif_sementes.add_prefix("semente_"))
print("\nDiferença Moderna - Clássica da fatia (pontos percentuais), por semente:")
print(estabilidade.sort_values("media_pp", ascending=False).round(2).to_string())

## 5. A hipótese da revolução dos três pontos

### 5.1 Peso do perímetro no modelo

Recorte das famílias ligadas ao arremesso de três: volume (`fg3ar`), aproveitamento
(`fg3_pct`, `efg`), criação das cestas de três (`pct_3p_assistidos`), defesa do perímetro
(`opp_fg3ar`) e o contraponto, a fração de pontos de meia distância (`pct_meia_distancia`). O
peso do perímetro é a soma das fatias dessas famílias, em cada era e em cada semente.

In [ ]:
presentes = [f for f in FAMILIAS_TRES if f in comp.index]
hipotese = comp.loc[presentes].join(ic[["ic_inf", "ic_sup"]]).join(
    estabilidade[["media_pp", "amplitude_pp"]].add_suffix("_sementes"))
print(hipotese.sort_values("diferenca", ascending=False).round(4).to_string())

total_c = hipotese["fatia_classica"].sum()
total_m = hipotese["fatia_moderna"].sum()
print(f"\nPeso conjunto das famílias de perímetro: {ERAS[0]} {total_c:.1%} -> "
      f"{ERAS[1]} {total_m:.1%} ({(total_m - total_c) * 100:+.1f} p.p.)")
for semente, f in fatias_semente.items():
    c, m = f.loc[presentes, ERAS[0]].sum(), f.loc[presentes, ERAS[1]].sum()
    print(f"  semente {semente}: {c:.1%} -> {m:.1%} ({(m - c) * 100:+.2f} p.p.)")

### 5.2 Trajetória temporada a temporada, sem modelo

A comparação acima parte de um corte em 2015, mas a adoção do arremesso de três foi gradual: o
3PAr da liga sobe de 0,196 em 2005 para 0,415 em 2026 sem degrau (etapa 1, seção 6). Para que o
resultado não dependa de onde a linha foi traçada, a pergunta é refeita **em cada uma das 20
temporadas e sem modelo nenhum**: a **AUC univariada** usa o próprio diferencial de uma
variável como pontuação e mede o quanto ele, sozinho, ordena corretamente as partidas daquela
temporada (0,5 = acaso). Como não há modelo ajustado, o resultado não pode ser artefato de
ajuste.

São quatro variáveis, todas na janela da temporada: o volume e o aproveitamento de três, o
aproveitamento efetivo (eFG%) e, como **referência**, o saldo de eficiência (`net_rtg`), que
mede a força geral da equipe. Se a referência caísse junto, a perda seria da liga inteira
ficando imprevisível, e não de uma variável específica.

A tendência de cada série é medida pelo ρ de Spearman entre a temporada e a AUC: negativo
quando a AUC tende a cair ao longo do tempo. A última tabela verifica se a tendência do volume
de três resiste a retirar a temporada do *lockout*, as duas últimas temporadas, ou a olhar cada
era separadamente.

In [ ]:
TRAJETORIA = {"dif_tmp_net_rtg": "força do time (saldo por posse)",
              "dif_tmp_fg3ar": "volume de 3 (3PAr)",
              "dif_tmp_fg3_pct": "aproveitamento de 3 (3P%)",
              "dif_tmp_efg": "eficiência de arremesso (eFG%)"}

linhas = []
for col, rotulo in TRAJETORIA.items():
    base = dados.dropna(subset=[col])
    # Orientação fixada no conjunto todo: uma temporada em que a relação se inverte
    # aparece abaixo de 0,5, e não como se tivesse ganho poder.
    sinal = 1 if roc_auc_score(base["alvo"], base[col]) >= 0.5 else -1
    for (temp, era), g in base.groupby(["temporada", "era"]):
        linhas.append({"variavel": rotulo, "coluna": col, "temporada": temp, "era": era,
                       "jogos": len(g), "auc": roc_auc_score(g["alvo"], sinal * g[col])})
traj = pd.DataFrame(linhas)

print("AUC univariada por temporada:")
print(traj.pivot_table(index="temporada", columns="variavel", values="auc").round(4).to_string())

print("\nTendência ao longo das 20 temporadas:")
for rotulo, g in traj.groupby("variavel", sort=False):
    rho, pv = spearmanr(g["temporada"], g["auc"])
    inclinacao = np.polyfit(g["temporada"], g["auc"], 1)[0]
    inicio = g.nsmallest(5, "temporada")["auc"].mean()
    fim = g.nlargest(5, "temporada")["auc"].mean()
    print(f"  {rotulo:32s} rho={rho:+.3f} (p={pv:.4f})  {inclinacao * 10:+.4f}/década  "
          f"5 primeiras {inicio:.4f} -> 5 últimas {fim:.4f}")

g = traj[traj["coluna"] == "dif_tmp_fg3ar"]
print("\nRobustez da tendência do volume de 3:")
for rotulo, sub in [("20 temporadas", g),
                    ("sem 2012 (lockout)", g[g["temporada"] != 2012]),
                    ("sem as 2 últimas", g[g["temporada"] <= g["temporada"].max() - 2]),
                    (f"só {ERAS[0]}", g[g["era"] == ERAS[0]]),
                    (f"só {ERAS[1]}", g[g["era"] == ERAS[1]])]:
    rho, pv = spearmanr(sub["temporada"], sub["auc"])
    print(f"  {rotulo:22s} n={len(sub):2d}  rho={rho:+.3f}  p={pv:.4f}")

### 5.3 Incerteza de cada ponto da trajetória

Cada AUC da trajetória foi estimada em cerca de mil partidas e carrega erro amostral: se as
partidas de uma temporada tivessem saído um pouco diferentes, o ponto cairia em outro lugar. O
ρ de Spearman acima trata os 20 pontos como valores exatos e não cobre essa incerteza.

Ela é medida por *bootstrap* das **partidas dentro de cada temporada**: 1.000 sorteios com
reposição, recalculando a AUC em cada um. A faixa de 95% de cada ponto é a que contém 95%
desses valores. Como cada sorteio produz uma trajetória completa de 20 pontos, o ρ também é
recalculado 1.000 vezes, o que dá um intervalo para a própria tendência e a proporção de
trajetórias em que ela é negativa.

In [ ]:
rng_traj = np.random.default_rng(SEMENTE)


def auc_reamostrada(y, x, b):
    # AUC de b reamostragens de uma vez, pela forma de postos (igual ao roc_auc_score).
    n = len(y)
    idx = rng_traj.integers(0, n, size=(b, n))
    postos = rankdata(x[idx], axis=1)
    ys = y[idx]
    pos = ys.sum(axis=1)
    return ((postos * ys).sum(axis=1) - pos * (pos + 1) / 2) / (pos * (n - pos))


ic_pontos, ic_rho = [], []
for col, rotulo in TRAJETORIA.items():
    base = dados.dropna(subset=[col])
    sinal = 1 if roc_auc_score(base["alvo"], base[col]) >= 0.5 else -1
    temps, replicas = [], []
    for temp, g in base.groupby("temporada"):
        a = auc_reamostrada(g["alvo"].to_numpy(), sinal * g[col].to_numpy(), B_TRAJETORIA)
        temps.append(temp)
        replicas.append(a)
        ic_pontos.append({"coluna": col, "temporada": temp,
                          "auc_ic_baixo": np.percentile(a, 2.5),
                          "auc_ic_alto": np.percentile(a, 97.5)})
    m = np.array(replicas)                     # temporadas × réplicas
    rhos = np.array([spearmanr(temps, m[:, b])[0] for b in range(B_TRAJETORIA)])
    ic_rho.append({"variavel": rotulo, "rho_ic_baixo": np.percentile(rhos, 2.5),
                   "rho_ic_alto": np.percentile(rhos, 97.5),
                   "replicas_com_rho_negativo": (rhos < 0).mean()})

traj = traj.merge(pd.DataFrame(ic_pontos), on=["coluna", "temporada"], how="left")

g = traj[traj["coluna"] == "dif_tmp_fg3ar"]
inclinacao = np.polyfit(g["temporada"], g["auc"], 1)[0]
largura = (traj["auc_ic_alto"] - traj["auc_ic_baixo"]).mean()
print(f"largura média da faixa de 95% de um ponto: {largura:.3f} de AUC")
print(f"queda da reta de tendência do volume de 3 entre {g['temporada'].min()} e "
      f"{g['temporada'].max()}: {-inclinacao * (g['temporada'].max() - g['temporada'].min()):.3f}")
print(f"\nTendência com a incerteza de cada ponto propagada ({B_TRAJETORIA} reamostragens):")
print(pd.DataFrame(ic_rho).set_index("variavel").round(3).to_string())

abaixo = g[g["auc"] < 0.5][["temporada", "auc", "auc_ic_baixo", "auc_ic_alto"]]
print("\nTemporadas em que a AUC do volume de 3 fica abaixo de 0,5:")
print(abaixo.round(3).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(1, len(TRAJETORIA), figsize=(15, 4.2), sharey=True,
                        facecolor="#fcfcfb")
for ax, (col, rotulo) in zip(axs, TRAJETORIA.items()):
    g = traj[traj["coluna"] == col].sort_values("temporada")
    ax.set_facecolor("#fcfcfb")
    ax.axhline(0.5, color="#b8b7b1", lw=1, ls="--")
    ax.fill_between(g["temporada"], g["auc_ic_baixo"], g["auc_ic_alto"],
                    color="#b8b7b1", alpha=0.35, lw=0)
    for era, gg in g.groupby("era"):
        ax.plot(gg["temporada"], gg["auc"], "o-", ms=4, lw=1.6,
                color=CORES_ERA[era], label=era)
    b, a = np.polyfit(g["temporada"], g["auc"], 1)
    xs = np.array([g["temporada"].min(), g["temporada"].max()])
    ax.plot(xs, a + b * xs, color="#3d3d3a", lw=1, ls=":")
    ax.set_title(rotulo, fontsize=10, color="#1a1a19")
    ax.grid(axis="y", color="#e5e4df", lw=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(CINZA)
    ax.tick_params(colors="#6b6a63", labelsize=9)
axs[0].set_ylabel("AUC univariada do diferencial", fontsize=10, color="#3d3d3a")
axs[0].legend(frameon=False, fontsize=9)
fig.suptitle("Poder de discriminação por temporada, com faixa de 95% do bootstrap",
             fontsize=12, color="#1a1a19")
plt.tight_layout()
plt.savefig(DIR_FIGURAS / "auc_univariada_temporada.png", dpi=150, facecolor="#fcfcfb")
plt.show()

### 5.4 Por que o volume de três perdeu poder

Três medidas, todas sem modelo, ajudam a interpretar a queda:

1. **AUC univariada de cada variável na era inteira**, para comparar com as fatias do SHAP.
2. **Correlação entre o diferencial de 3PAr e o diferencial de eficiência ofensiva**
   (`ortg`): mede o quanto arremessar mais de três que o adversário acompanha atacar melhor
   que ele.
3. **Dispersão do 3PAr entre as equipes**, medida dentro de cada temporada (desvio-padrão do
   3PAr das 30 equipes) e depois resumida por era, ao lado do desvio-padrão do diferencial. A
   medida é feita dentro da temporada porque o 3PAr médio da liga sobe ao longo de cada era, e
   uma dispersão calculada sobre a era inteira misturaria essa subida com a diferença entre
   equipes. O desvio relativo divide pelo 3PAr médio da liga na temporada.

Se o volume de três perdeu poder sem que as equipes tenham ficado mais parecidas entre si, a
perda vem de o volume ter deixado de acompanhar a qualidade do ataque.

In [ ]:
UNIVARIADAS = {"dif_tmp_net_rtg": "net_rtg", "dif_tmp_fg3ar": "fg3ar",
               "dif_tmp_fg3_pct": "fg3_pct", "dif_tmp_efg": "efg", "dif_tmp_opp_efg": "opp_efg"}
linhas = []
for col, nome in UNIVARIADAS.items():
    base = dados.dropna(subset=[col])
    sinal = 1 if roc_auc_score(base["alvo"], base[col]) >= 0.5 else -1
    linhas.append({"familia": nome, **{era: roc_auc_score(g["alvo"], sinal * g[col])
                                       for era, g in base.groupby("era")}})
auc_era = pd.DataFrame(linhas).set_index("familia")
auc_era["diferenca"] = auc_era[ERAS[1]] - auc_era[ERAS[0]]
print("(1) AUC univariada do diferencial na era inteira:")
print(auc_era.round(4).to_string())

print("\n(2) Correlação entre o diferencial de 3PAr e o de eficiência ofensiva:")
print(dados.groupby("era")[["dif_tmp_fg3ar", "dif_tmp_ortg"]]
      .apply(lambda g: g["dif_tmp_fg3ar"].corr(g["dif_tmp_ortg"])).round(3).to_string())

times = pd.read_parquet(DIR_DADOS / "partidas_times.parquet",
                        columns=["era", "temporada", "team_id", "fg3a", "fga"])
tpar = (times.groupby(["era", "temporada", "team_id"])[["fg3a", "fga"]].sum()
        .eval("fg3a / fga").rename("tpar").reset_index())
por_temporada = tpar.groupby(["era", "temporada"])["tpar"].agg(["mean", "std"])
print("\n(3) Dispersão do 3PAr entre as equipes (média das temporadas de cada era):")
print(pd.DataFrame({
    "3PAr_medio_da_liga": por_temporada["mean"].groupby("era").mean(),
    "dp_entre_equipes": por_temporada["std"].groupby("era").mean(),
    "dp_relativo": (por_temporada["std"] / por_temporada["mean"]).groupby("era").mean(),
    "dp_do_diferencial": dados.groupby("era")["dif_tmp_fg3ar"].std(),
}).round(4).to_string())

## 6. Explicações locais (ilustrativas)

O gráfico de cascata (*waterfall*) mostra como a previsão de **uma** partida parte do valor-base
e chega à probabilidade final, variável por variável. São três casos por era: um acerto
confiante, um erro confiante e um jogo equilibrado.

Os casos são escolhidos pela probabilidade **fora da amostra** (a do teste da etapa 3, quando o
modelo ainda não tinha visto a partida), mas a cascata decompõe o **modelo final**, ajustado na
era inteira, que já viu essas partidas. As duas probabilidades diferem, e o título mostra as
duas. Os gráficos ilustram como a decomposição funciona; não são uma auditoria dos erros da
validação.

In [ ]:
prev = pd.read_parquet(DIR_RESULTADOS / "previsoes.parquet")
prev = prev[prev["modelo"] == ALGORITMO]

for era in ERAS:
    p = prev[prev["era"] == era].copy()
    p["erro"] = (p["prob"] - p["alvo"]).abs()
    casos = {
        "acerto confiante": p.nsmallest(1, "erro").iloc[0],
        "erro confiante": p.nlargest(1, "erro").iloc[0],
        "jogo equilibrado": p.iloc[(p["prob"] - 0.5).abs().argsort()].iloc[0],
    }
    posicao = {g: i for i, g in enumerate(ids_jogos[era])}
    for rotulo, linha in casos.items():
        i = posicao[int(linha["game_id"])]
        exp = shap.Explanation(values=shap_vals[era][i], base_values=base_vals[era][i],
                               data=matrizes[era][i], feature_names=FEATURES)
        prob_final = float(base_vals[era][i] + shap_vals[era][i].sum())
        shap.plots.waterfall(exp, max_display=12, show=False)
        plt.title(f"{era} — {rotulo} (jogo {int(linha['game_id'])}, real {int(linha['alvo'])})"
                  f"\nprob. modelo final {prob_final:.2f} | prob. fora da amostra "
                  f"{linha['prob']:.2f}", fontsize=11)
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / f"waterfall_{era}_{rotulo.replace(' ', '_')}.png",
                    dpi=160, bbox_inches="tight")
        plt.show()

## 7. Salvar

In [ ]:
pd.DataFrame({
    "familia": fam,
    f"fatia_{ERAS[0]}": fatia[ERAS[0]], f"fatia_{ERAS[1]}": fatia[ERAS[1]],
    f"abs_shap_{ERAS[0]}": importancia[ERAS[0]], f"abs_shap_{ERAS[1]}": importancia[ERAS[1]],
}).sort_values(f"fatia_{ERAS[1]}", ascending=False).to_csv(
    DIR_RESULTADOS / "shap_por_feature.csv")
comp.join(ic[["ic_inf", "ic_sup", "significativo"]]).sort_values(
    "diferenca", ascending=False).to_csv(DIR_RESULTADOS / "shap_por_familia.csv")
estabilidade.to_csv(DIR_RESULTADOS / "shap_sementes.csv")
traj.to_csv(DIR_RESULTADOS / "auc_univariada_temporada.csv", index=False)

(DIR_RESULTADOS / "metadados_shap.json").write_text(json.dumps({
    "algoritmo": ALGORITMO,
    "eras": {era: int(shap_vals[era].shape[0]) for era in ERAS},
    "n_features": len(FEATURES), "n_familias": int(len(comp)),
    "aditividade_erro_maximo": erros_adit,
    "spearman_entre_eras": {"rho": float(rho_eras), "p": float(p_eras)},
    "bootstrap_temporadas": {"n": N_BOOTSTRAP, "familias_com_ic_sem_zero":
                             int((ic["significativo"] == "sim").sum())},
    "sementes": [SEMENTE, *SEMENTES_EXTRAS],
    "peso_perimetro": {ERAS[0]: float(total_c), ERAS[1]: float(total_m)},
    "bootstrap_trajetoria": {"n": B_TRAJETORIA, "largura_media_ic_ponto": float(largura)},
}, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"tabelas em {DIR_RESULTADOS}/ e figuras em {DIR_FIGURAS}/")